# WEEK 3 — MODEL OPTIMIZATION AND UNSUPERVISED LEARNING

**ML-Powered Customer Analytics**    
**Program:** BS Electronics  
**Department:** Department of Electronics  
**University:** Quaid-i-Azam University, Islamabad  
**Course:** Introduction to Applied AI  
**Lab:** Lab 3 — Week 3  
**Date:** 04 October 2026   

---

### Objective
The objective of this lab is to measure and minimize split noise, optimize model hyperparameters, and explore customer structures through unsupervised learning on the Telco Customer Churn dataset. The analysis covers measuring train/validation split noise, 5-fold cross-validation, hyperparameter tuning using Grid Search and Random Search for Random Forest and XGBoost (with early stopping), customer segmentation using K-Means clustering, variance analysis and dimensionality reduction via Principal Component Analysis (PCA), and final model selection evaluated strictly once on a locked test set.

---

### Dataset
The analysis uses the preprocessed Telco Customer Churn dataset, containing 7,043 customer records and 30 features after one-hot encoding. A stratified 20% test set (1,409 rows) is locked away and evaluated only at the final pipeline step.

---

### Evaluation Focus
The models and methods are evaluated using:

* Stratified 5-Fold Cross-Validation (Mean ± Std)
* Standard Error ($\sqrt{p(1-p)/n}$) and Variance Analysis
* ROC-AUC, Recall, Precision, and F1-score
* Hyperparameter Tuning (Grid Search vs. Random Search)
* Validation Curves and Loss Convergence (Early Stopping)
* K-Means Inertia (WCSS) and Silhouette Scores
* Principal Component Loadings and Scree Variance Analysis
* Single-test evaluation for honest generalization assessment

---

## Haider Ali Asghar  
**Date:** 04/10/2026

### Task 1.1: Imports

In [ ]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import loguniform, randint, uniform
from sklearn.model_selection import (train_test_split, StratifiedKFold,
                                     cross_validate, validation_curve,
                                     GridSearchCV, RandomizedSearchCV)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import (accuracy_score, roc_auc_score, recall_score,
                             precision_score, f1_score, silhouette_score)
from xgboost import XGBClassifier
import joblib

**What this cell does:**  
Loads all the libraries required for the analysis: `pandas` and `numpy` for data manipulation, `matplotlib` for plotting, `scikit-learn` for models, cross-validation, and clustering, `XGBoost` for gradient boosting, and `joblib` for saving the final model pipeline.

### Task 1.2: Load, encode, and lock away the test set


In [ ]:
path = '/kaggle/input/datasets/blastchar/telco-customer-churn/WA_Fn-UseC_-Telco-Customer-Churn.csv'
df = pd.read_csv(path)
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df['TotalCharges'] = df['TotalCharges'].fillna(0)   # blanks have tenure = 0
def build_features(data):
    X = data.drop(columns=['customerID', 'Churn'])
    return pd.get_dummies(X, drop_first=True).astype(float)
y = (df['Churn'] == 'Yes').astype(int)
X = build_features(df)
# The test set is locked away until Part 7. Do not touch it before then.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)
print(X_train.shape, X_test.shape)

**What this cell does:**  
- Reads the Telco churn CSV dataset and converts `TotalCharges` into numeric values.
- Fills the 11 blank `TotalCharges` values with `0` (since those customers have `tenure = 0` and have not paid anything yet). This avoids data leakage because no statistical imputation is used.
- Converts categorical text columns into 0/1 binary features via one-hot encoding, set `Churn` as the target `y` (1 = churned).
- Splits the dataset 80/20 with stratification. The 20% test set (1,409 rows) is locked away until Part 7.
- **Output shape:** Train set `(5634, 30)`, Test set `(1409, 30)`.

### Task 1.3: Same model, twenty different splits

In [ ]:
lr = Pipeline([('scale', StandardScaler()),
               ('model', LogisticRegression(max_iter=1000))])
scores = []
for seed in range(20):
    Xa, Xb, ya, yb = train_test_split(X_train, y_train, test_size=0.25,
                                      random_state=seed, stratify=y_train)
    scores.append(accuracy_score(yb, lr.fit(Xa, ya).predict(Xb)))
scores = np.array(scores)
print(f'min {scores.min():.3f}  max {scores.max():.3f}  std {scores.std():.4f}')
p, n = scores.mean(), len(yb)
se = np.sqrt(p * (1 - p) / n)
print(f'Theoretical standard error: {se:.4f}  -> 95% CI +/- {1.96 * se:.3f}')
plt.bar(range(20), scores)
plt.ylim(scores.min() - 0.02, scores.max() + 0.02)
plt.xlabel('random_state'); plt.ylabel('Validation accuracy')
plt.title('Same model, same data, different split'); plt.show()

### Split Noise and Why Cross-Validation Matters

The purpose of this experiment is to measure how much the reported performance of the same Logistic Regression model can change when only the random train-validation split is changed. The model, dataset, and evaluation method remain the same, while 20 different random seeds are used.

Across the 20 seeds, the validation accuracy ranged from **78.0% to 82.8%**, with a standard deviation of **0.0104**. The theoretical standard error was **0.0107**, giving an approximate 95% interval of **±0.021**, or about **±2.1 percentage points**.

The results show that a single train-validation split can introduce noticeable variation in the reported accuracy. Therefore, if two students compared Logistic Regression and Random Forest using different random seeds, they could potentially reach opposite conclusions when the difference between the models is small. For example, a change of several percentage points caused only by the split could make one model appear better in one experiment and another model appear better in a different split.

This is why relying on a single validation split is not a reliable way to compare models. Cross-validation reduces this split-to-split uncertainty by evaluating the model on several different validation folds and reporting the mean and standard deviation.


## Part 2: 5-Fold Cross-Validation 

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
metrics = ['roc_auc', 'recall', 'f1']
def cv_report(name, model):
    r = cross_validate(model, X_train, y_train, cv=cv, scoring=metrics)
    row = {'model': name}
    for m in metrics:
        s = r['test_' + m]
        row[m] = f'{s.mean():.3f} +/- {s.std():.3f}'
    return row
rf = RandomForestClassifier(n_estimators=300, min_samples_leaf=5,
                            n_jobs=-1, random_state=42)
print(pd.DataFrame([cv_report('Logistic Regression', lr),
                    cv_report('Random Forest', rf)]).to_string(index=False))

### 5-Fold Cross-Validation Results

The 5-fold cross-validation experiment evaluates Logistic Regression and Random Forest on the training data using the same five stratified folds. The preprocessing pipeline is refitted inside each training fold, which prevents information from the validation fold from leaking into the preprocessing stage.

| Model               |           ROC-AUC |            Recall |                F1 |
| ------------------- | ----------------: | ----------------: | ----------------: |
| Logistic Regression | **0.846 ± 0.013** | **0.545 ± 0.042** | **0.594 ± 0.030** |
| Random Forest       | **0.844 ± 0.011** | **0.496 ± 0.019** | **0.573 ± 0.020** |

The ROC-AUC difference between Logistic Regression and Random Forest is only **0.002**. This difference is much smaller than the standard deviation of either model, so the results do not provide strong evidence that Logistic Regression is meaningfully better than Random Forest based on AUC alone.

The recall results provide additional information. Logistic Regression achieved a mean recall of **0.545**, while Random Forest achieved **0.496**. Therefore, at the default classification threshold, Logistic Regression identified a larger proportion of the actual churners.

AUC and recall measure different aspects of performance. **AUC** evaluates how well the model ranks churners above non-churners across different classification thresholds, whereas **recall** measures the proportion of actual churners correctly identified at the selected threshold. Therefore, a model can have a similar AUC but still have different recall at the operating threshold.

Overall, the cross-validation results show that the two models have very similar ranking performance, while Logistic Regression has an advantage in recall and F1 at the default threshold.


## Part 3: Hyperparameter Tuning

### Task 3.1: Validation curve for Logistic Regression's C

In [ ]:
Cs = np.logspace(-4, 2, 13)
tr, va = validation_curve(lr, X_train, y_train, param_name='model__C',
                          param_range=Cs, cv=cv, scoring='roc_auc')
plt.semilogx(Cs, tr.mean(axis=1), 'o-', label='Train AUC')
plt.semilogx(Cs, va.mean(axis=1), 'o-', label='CV AUC')
plt.xlabel('C  (smaller = stronger regularization)'); plt.ylabel('AUC')
plt.legend(); plt.title('Validation curve: Logistic Regression'); plt.show()
print('Best C by CV:', Cs[va.mean(axis=1).argmax()])

**What this cell does:**  
Trains Logistic Regression across 13 values of `C` ranging from $0.0001$ to $100$ (log scale) inside a Pipeline (`model__C`) and evaluates both Train AUC and 5-fold CV AUC for each value.

**Graph context:**  
The x-axis displays `C` on a log scale, while the y-axis tracks AUC. The orange curve represents CV AUC and the blue curve represents Train AUC. At low `C` values (left side), severe regularization causes underfitting. As `C` grows, performance rises and plateaus. The close distance between train and CV lines demonstrates minimal overfitting.

**Insight:**  
From the validation curve, underfitting ends when C is increased from the very small values on the left (roughly C ≈ 0.01 to 0.1). After that the CV AUC reaches a plateau. The best C is 10, but the curve is flat there, so C = 1 or 10 give practically the same result. Train and CV AUC stay close, so LR is not overfitting.

### Task 3.2: Grid search for Random Forest

In [ ]:
grid = {'max_depth': [4, 8, 12, None],
        'min_samples_leaf': [1, 5, 20],
        'max_features': ['sqrt', 0.5]}
print('Combinations:', 4 * 3 * 2, ' Fits:', 4 * 3 * 2 * 5)
t0 = time.time()
gs = GridSearchCV(RandomForestClassifier(n_estimators=200, n_jobs=-1,
                                         random_state=42),
                  grid, cv=cv, scoring='roc_auc')
gs.fit(X_train, y_train)
print(f'Grid:   best AUC {gs.best_score_:.4f}  in {time.time() - t0:.0f}s')
print(gs.best_params_)

### Task 3.3: Random search for Random Forest

In [ ]:
dist = {'max_depth': randint(3, 20),
        'min_samples_leaf': randint(1, 40),
        'max_features': uniform(0.1, 0.8)}   # fraction of features
t0 = time.time()
rs = RandomizedSearchCV(RandomForestClassifier(n_estimators=200, n_jobs=-1,
                                               random_state=42),
                        dist, n_iter=24, cv=cv, scoring='roc_auc',
                        random_state=42)
rs.fit(X_train, y_train)
print(f'Random: best AUC {rs.best_score_:.4f}  in {time.time() - t0:.0f}s')
print(rs.best_params_)
res = pd.DataFrame(rs.cv_results_)
cols = ['param_max_depth', 'param_min_samples_leaf', 'mean_test_score',
        'std_test_score', 'rank_test_score']
print(res[cols].sort_values('rank_test_score').head(5).to_string(index=False))

### Grid Search vs Random Search

The Random Forest grid search evaluated **24 hyperparameter combinations** using 5-fold cross-validation, resulting in **120 total fits**. It achieved a best CV AUC of **0.8468**.

The RandomizedSearchCV experiment also evaluated **24 configurations with 5-fold cross-validation**, resulting in **120 total fits**. It achieved a best CV AUC of **0.8464**.

In this experiment, grid search produced a slightly higher best CV AUC than random search:

* **Grid search:** 0.8468
* **Random search:** 0.8464
* **Total fits:** 120 for each method

The grid search took approximately **69 seconds**, while the random search took approximately **74 seconds** in this experiment. Therefore, grid search was slightly faster and also produced a slightly higher best CV score for this particular search space.

However, the choice can change when the hyperparameter space becomes larger. If I had to tune six hyperparameters, I would generally prefer **random search** because it can explore a wider range of combinations without requiring every possible combination to be evaluated. This becomes more efficient when only a few hyperparameters have a strong effect on model performance.

The best CV scores from grid search and random search should not be treated as final unbiased performance because they are the best results selected from many tested configurations. This creates a risk of optimistic selection bias. The final model therefore needs to be compared again using the common cross-validation procedure in Part 7 before the test set is used.

## Part 4: XGBoost

### Task 4.1: Early stopping on a validation split

In [ ]:
X_tr, X_val, y_tr, y_val = train_test_split(
    X_train, y_train, test_size=0.2, random_state=42, stratify=y_train)
spw = (y_tr == 0).sum() / (y_tr == 1).sum()
print(f'scale_pos_weight = {spw:.2f}')
xgb = XGBClassifier(n_estimators=2000, learning_rate=0.03, max_depth=4,
                    subsample=0.8, colsample_bytree=0.8,
                    eval_metric='logloss', early_stopping_rounds=100,
                    random_state=42, n_jobs=-1)
xgb.fit(X_tr, y_tr, eval_set=[(X_tr, y_tr), (X_val, y_val)], verbose=False)
print('Best number of trees:', xgb.best_iteration + 1)
val_auc = roc_auc_score(y_val, xgb.predict_proba(X_val)[:, 1])
print(f'Validation AUC: {val_auc:.4f}')

### XGBoost Early Stopping

The initial XGBoost model was allowed to train for up to 2,000 boosting rounds, but early stopping selected **247 trees** as the best iteration. The validation AUC reached **0.8541**.

The training loss continued to decrease as more trees were added, while the validation loss eventually stopped improving. This indicates that additional trees were beginning to fit patterns specific to the training data rather than improving generalization.

Early stopping therefore helped control overfitting by stopping the boosting process when validation performance no longer improved.


### Task 4.2: Plot train vs validation loss

In [ ]:
ev = xgb.evals_result()
plt.plot(ev['validation_0']['logloss'], label='Train log-loss')
plt.plot(ev['validation_1']['logloss'], label='Validation log-loss')
plt.axvline(xgb.best_iteration, color='k', ls='--', label='Early stop')
plt.xlabel('Boosting round'); plt.ylabel('Log-loss')
plt.legend(); plt.title('XGBoost: when to stop adding trees'); plt.show()

**What this cell does:**  
Plots train log-loss vs. validation log-loss across boosting rounds, displaying a vertical dashed line where early stopping identified minimum validation loss.

**Graph context:**  
The blue line (train loss) steadily decreases as trees are added. The orange line (validation loss) drops, achieves its minimum near round 246 (dashed line), and then slowly rises. Training beyond this marker results in memorizing training noise (overfitting).

### Task 4.3: Random search over XGBoost hyperparameters

In [ ]:
xdist = {'max_depth': randint(2, 7),
         'learning_rate': loguniform(0.01, 0.3),
         'n_estimators': randint(100, 600),
         'subsample': uniform(0.6, 0.4),
         'colsample_bytree': uniform(0.5, 0.5),
         'min_child_weight': randint(1, 10),
         'reg_lambda': loguniform(0.1, 10)}
xs = RandomizedSearchCV(XGBClassifier(eval_metric='logloss', random_state=42,
                                      n_jobs=-1),
                        xdist, n_iter=30, cv=cv, scoring='roc_auc',
                        random_state=42)
xs.fit(X_train, y_train)
print(f'XGBoost tuned CV AUC: {xs.best_score_:.4f}')
print({k: round(v, 3) if isinstance(v, float) else v
       for k, v in xs.best_params_.items()})

### Task 4.4: What did XGBoost use?

In [ ]:
imp = pd.Series(xs.best_estimator_.feature_importances_, index=X.columns)
imp.sort_values().tail(10).plot.barh(title='XGBoost feature importance (gain)')
plt.show()

### Tuned XGBoost Performance

The tuned XGBoost model achieved a **5-fold cross-validated AUC of 0.8502 ± 0.0117**.

Compared with the Logistic Regression and Random Forest results, XGBoost achieved the highest mean CV AUC. However, its improvement was relatively small and did not exceed one standard deviation. Therefore, XGBoost showed a modest advantage rather than a decisive improvement over the other models.

This result demonstrates why both the mean performance and its variation across folds should be considered when comparing machine learning models.


## Part 5: K-Means Customer Segmentation

### Task 5.1: Build and scale the segmentation features

In [ ]:
seg_cols = ['tenure', 'MonthlyCharges', 'TotalCharges']
svc = ['PhoneService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
       'TechSupport', 'StreamingTV', 'StreamingMovies']
seg = df[seg_cols].copy()
seg['n_services'] = (df[svc] == 'Yes').sum(axis=1)
Z = StandardScaler().fit_transform(seg)   # distances need equal scales
print(seg.describe().round(1))

### Task 5.2: Choose k with the elbow and silhouette

In [ ]:
ks = range(2, 9)
inertia, sil = [], []
for k in ks:
    km = KMeans(n_clusters=k, n_init=10, random_state=42).fit(Z)
    inertia.append(km.inertia_)
    sil.append(silhouette_score(Z, km.labels_, sample_size=3000,
                                random_state=42))
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(ks, inertia, 'o-'); ax[0].set_title('Elbow: inertia (WCSS)')
ax[1].plot(ks, sil, 'o-'); ax[1].set_title('Silhouette score')
for a in ax: a.set_xlabel('k')
plt.show()

### Task 5.3: Fit, then profile the segments

In [ ]:
k = 4   # choose from the elbow and silhouette plots, then justify it
km = KMeans(n_clusters=k, n_init=10, random_state=42).fit(Z)
seg['cluster'] = km.labels_
# Churn was NOT used to form clusters. We use it only to profile them.
profile = seg.assign(churn=y).groupby('cluster').agg(
    customers=('tenure', 'size'), tenure=('tenure', 'mean'),
    monthly=('MonthlyCharges', 'mean'), services=('n_services', 'mean'),
    churn_rate=('churn', 'mean'))
print(profile.round(2).sort_values('churn_rate', ascending=False))

### Customer Segmentation

The customer segmentation used tenure, MonthlyCharges, TotalCharges, and the number of subscribed services. The features were standardized before applying K-Means so that variables with larger numerical scales would not dominate the distance calculations.

The selected value was **k = 4**, providing four customer groups that can be interpreted in a meaningful business context. Churn was not used to create the clusters; it was only used afterward to profile the resulting segments.

| Segment                         | Customers | Avg. Tenure | Avg. Monthly Charge | Avg. Services | Churn Rate |
| ------------------------------- | --------: | ----------: | ------------------: | ------------: | ---------: |
| High-Value At-Risk Customers    |     2,157 |       18.38 |               80.41 |          3.28 |    **43%** |
| New and Light-Service Customers |     1,918 |        8.96 |               37.71 |          1.20 |    **32%** |
| Loyal High-Value Customers      |     1,938 |       59.83 |               92.09 |          5.06 |    **14%** |
| Stable Low-Cost Customers       |     1,030 |       53.61 |               30.96 |          1.48 |     **5%** |

The **High-Value At-Risk Customers** segment has the highest churn rate at **43%**. These customers should receive the highest retention priority through targeted retention offers, contract incentives, personalized support, and service-quality follow-ups.

The **New and Light-Service Customers** have a **32%** churn rate and relatively short tenure. Early onboarding, proactive support, and incentives to adopt additional services could help improve retention.

The **Loyal High-Value Customers** have a relatively low churn rate of **14%**, despite their high monthly charges and large number of services. Loyalty benefits and premium support would be appropriate for this group.

The **Stable Low-Cost Customers** have the lowest churn rate at only **5%**. A low-cost retention strategy such as regular communication and basic loyalty incentives would be appropriate.

Overall, the clusters show that customer risk differs substantially across combinations of tenure, spending, and service adoption.


## Part 6: Principal Component Analysis

### Task 6.1: Scree plot

In [ ]:
Xs = StandardScaler().fit_transform(X_train)
pca = PCA().fit(Xs)
cum = np.cumsum(pca.explained_variance_ratio_)
print('Components for 90% variance:', np.argmax(cum >= 0.90) + 1,
      'of', X_train.shape[1])
plt.bar(range(1, 16), pca.explained_variance_ratio_[:15], label='Per component')
plt.plot(range(1, 16), cum[:15], 'o-', color='C1', label='Cumulative')
plt.xlabel('Principal component'); plt.ylabel('Explained variance ratio')
plt.legend(); plt.title('Scree plot'); plt.show()

### Task 6.2: Customers in two dimensions, and PC1 loadings

In [ ]:
P = PCA(n_components=2).fit(Xs)
T = P.transform(Xs)
plt.figure(figsize=(7, 5))
plt.scatter(T[:, 0], T[:, 1], c=y_train, cmap='coolwarm', s=6, alpha=0.5)
plt.xlabel('PC1'); plt.ylabel('PC2')
plt.title('Customers in 2D, colored by churn (red = churned)'); plt.show()
load = pd.Series(P.components_[0], index=X.columns)
print('PC1 top loadings:')
order = load.abs().sort_values(ascending=False).index
print(load.reindex(order).head(6).round(3))

## Part 6 — Write It: PCA

PCA found that **15 of the 30 components are required to explain 90% of the variance** in the standardized training features. Therefore, the original 30-dimensional feature space contains substantial correlation and redundancy, but it is not reducible to only a very small number of dimensions without losing important information.

The strongest PC1 loadings were:

* InternetService_No: **0.302**
* OnlineSecurity_No internet service: **0.302**
* TechSupport_No internet service: **0.302**
* StreamingTV_No internet service: **0.302**
* DeviceProtection_No internet service: **0.302**
* OnlineBackup_No internet service: **0.302**

These identical loadings reveal an important redundancy in the one-hot encoded data. Several “No internet service” variables contain essentially the same information as `InternetService_No`. PCA is therefore detecting structure created by the encoding scheme rather than completely independent customer characteristics.

In the 2D PCA visualization, churners and stayers show substantial overlap rather than forming two perfectly separate groups. This suggests that churn is not a simple linearly separable problem in just two principal-component dimensions. Customer churn depends on several interacting characteristics, which explains why classification models can achieve good AUC while still making some errors.

---

## Part 7: Choose by CV, Test Once, Save

### Task 7.1: Final cross-validated comparison

In [ ]:
candidates = {
    'LR (tuned C)': lr.set_params(model__C=Cs[va.mean(axis=1).argmax()]),
    'RF (random search)': rs.best_estimator_,
    'XGBoost (tuned)': xs.best_estimator_,
}
rows = []
for name, m in candidates.items():
    cvs = cross_validate(m, X_train, y_train, cv=cv, scoring='roc_auc')
    rows.append({'model': name,
                 'cv_auc': cvs['test_score'].mean(),
                 'cv_std': cvs['test_score'].std()})
cmp = pd.DataFrame(rows)
print(cmp.round(4).to_string(index=False))

### Task 7.2: Unlock the test set, exactly once

In [ ]:
best_name = cmp.loc[cmp['cv_auc'].idxmax(), 'model']  # chosen by CV only
best = candidates[best_name].fit(X_train, y_train)
prob = best.predict_proba(X_test)[:, 1]
pred = (prob >= 0.5).astype(int)
print(f'{best_name} on the test set (used once):')
print(f'AUC {roc_auc_score(y_test, prob):.4f}  '
      f'recall {recall_score(y_test, pred):.3f}  '
      f'precision {precision_score(y_test, pred):.3f}')
joblib.dump(best, 'churn_model.joblib')   # you will deploy this in Week 4

## Part 7 — Write It: Final Model

The final model was selected using **5-fold cross-validated AUC on the training set**, while the test set remained locked until the final evaluation.

| Model              |     CV AUC |     CV Std |
| ------------------ | ---------: | ---------: |
| LR (tuned C)       | **0.8464** | **0.0129** |
| RF (random search) | **0.8464** | **0.0114** |
| XGBoost (tuned)    | **0.8502** | **0.0117** |

Based on the CV results, **XGBoost (tuned)** was selected because it achieved the highest mean CV AUC of **0.8502**.

The model was then fitted on the complete training set and evaluated on the previously untouched test set exactly once. It achieved:

* **Test AUC: 0.8483**
* **Recall: 0.521**
* **Precision: 0.659**

The test AUC of **0.8483** is within the approximate CV mean ± 2 standard deviations range of **0.8268 to 0.8736**, so the difference between CV and test performance is consistent with normal variation rather than evidence of severe overfitting.

Compared with the Week 2 best AUC of approximately **0.842**, the tuned XGBoost CV AUC of **0.8502** represents an improvement of about **0.0082 AUC**. On the final test set, the AUC of **0.8483** is approximately **0.0063 higher** than the Week 2 result. Therefore, tuning helped, but the improvement was modest rather than dramatic.

Finally, the trained model was saved as `churn_model.joblib` for use in Week 4.


## Check Your Understanding

### 1. Model A has 0.812 accuracy and Model B has 0.803 accuracy on the same 1,409 test rows. Is Model A definitely better?

No. A difference of 0.009 in accuracy is relatively small and should be interpreted considering sampling uncertainty. With the same test size, the standard error of accuracy depends on the observed accuracy. For an accuracy around 0.81, the approximate standard error is about 0.0105, so the 0.009 difference is smaller than one standard error. Therefore, we should not automatically conclude that Model A is meaningfully better.

### 2. Why should the scaler be inside a Pipeline during cross-validation?

The scaler should be inside the Pipeline so that it is fitted only on the training portion of each CV fold. If scaling is performed before cross-validation, information from the validation fold can influence the scaling parameters, creating data leakage. Putting the scaler inside the Pipeline ensures that every fold is processed independently and fairly.

### 3. A grid has 5 values for one parameter, 4 for another, and 3 for a third. With 5-fold CV, how many fits are required?

There are:

**5 × 4 × 3 = 60 hyperparameter combinations**

With 5-fold cross-validation:

**60 × 5 = 300 fits**

Therefore, the search requires **300 model fits**.

### 4. Why is `gs.best_score_` optimistic?

`gs.best_score_` is optimistic because it is the best score among many configurations that were tested. Even if all configurations have similar true performance, one may obtain an unusually high CV score simply through random variation. Selecting that winner introduces a form of selection bias known as the winner's curse. A final unbiased estimate should therefore come from data that was not used to select the model, such as the locked test set.

### 5. What does each new tree in gradient boosting try to predict?

Each new tree is trained to correct the errors made by the existing ensemble. More precisely, it approximates the residuals or negative gradient of the loss function with respect to the current predictions. By repeatedly correcting previous errors, the ensemble gradually improves its predictions.

### 6. Why would unscaled K-Means clusters be dominated by TotalCharges?

K-Means uses distances between observations. TotalCharges has a much larger numerical range than variables such as tenure, MonthlyCharges, or the number of services. Without scaling, differences in TotalCharges would contribute much more to the distance calculation. As a result, the clusters could mostly reflect TotalCharges rather than a balanced combination of customer characteristics. Standardization gives the variables comparable influence.
